# Moon val revision audit

## tl;dr

The corrected native CVAT ZIP preserves all 33 Moon source-val frames: 96 boxes versus 106, and zero uncertain versus 13. The human explicitly confirmed that all ten deleted candidates are non-craters. Three remaining objects were confirmed and one truncated bbox refined. The other exports and three specifically queried rim flags were confirmed unchanged. Current v4 has **397 invariant candidate answers and 3 unknown references out of 400; zero accepted scoring targets**. The gray source band in mars_seg_018 is retained as an additional native artifact, not generated E2 corruption.

## Context & Methods

Read-only native-CVAT comparison against immutable v3, v4 integrity, human-decision traceability and source-artifact checks. Object order and XML indices are not identity.

### Key Assumptions

Deletion alone is not absence evidence. After an explicit question about the ten deleted candidates, the human answered: �����������, ��� ��� �� ��������. The separate statement ��������� ������ �����. ��� ��������� ����������, �������� confirms the unchanged exports and three queried clipping flags, not independent completeness/identity review of every recovered publisher object. The later instruction ��� ���� ������. ������ �� ����� ��� ��� ��������� retains original pixels without clearing unknown references. Existing clean labels mean unmodified source, not artifact-free. All ordinary Python cells are executed sequentially with captured stdout; this is not a Jupyter-kernel run.

## Data

### 1. Resolve bounded local inputs

In [1]:
from pathlib import Path
from collections import Counter
import json, sys, tomllib
from PIL import Image
WORKSPACE=next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'src/planetary_vlm').is_dir() and (p/'data_orbital').is_dir())
sys.path.insert(0,str(WORKSPACE/'src'))
from planetary_vlm.datasets.crater_annotations import import_export, verify_crater_build
from planetary_vlm.datasets.orbital_questions import file_hash
from planetary_vlm.io import read_jsonl
OLD=WORKSPACE/'data_orbital/crater_annotations_v3/exports/moon_val.zip'
NEW=Path(r'C:\Users\Timur\Downloads\moon_craters_val_v1 (1).zip')
assert file_hash(OLD)=='6bf2791ca7db26ef53503a1843a1c3e4e4fcb04c666101f4c173b78305c80f3f'
assert file_hash(NEW)=='664990e98e673eb8b51f8b81c21db3fd8a6da16d6b29eab2454529ad9d47daf0'
print(json.dumps({'old':str(OLD),'old_sha256':file_hash(OLD),'new':str(NEW),'new_sha256':file_hash(NEW)},indent=2))

{
  "old": "C:\\Users\\Timur\\myprojects\\robo-projects\\article\\data_orbital\\crater_annotations_v3\\exports\\moon_val.zip",
  "old_sha256": "6bf2791ca7db26ef53503a1843a1c3e4e4fcb04c666101f4c173b78305c80f3f",
  "new": "C:\\Users\\Timur\\Downloads\\moon_craters_val_v1 (1).zip",
  "new_sha256": "664990e98e673eb8b51f8b81c21db3fd8a6da16d6b29eab2454529ad9d47daf0"
}


### 2. Validate both native CVAT exports

In [2]:
rows={r['sample_id']:r for r in read_jsonl(WORKSPACE/'data_orbital/orbital_300_v3/moon/segmentation_manifest.jsonl') if r['source_split']=='val'}
rules=tomllib.loads((WORKSPACE/'configs/crater_annotation_v2.toml').read_text(encoding='utf-8'))
dimensions={}
for sid,row in rows.items():
    with Image.open(WORKSPACE/row['image']) as im: dimensions[sid]=im.size
old,_,old_record=import_export(OLD,rows,dimensions,rules)
new,_,new_record=import_export(NEW,rows,dimensions,rules)
assert len(old)==len(new)==33
assert all(r['complete_for_size_rule'] for r in new.values())
profile={}
for name,frames in [('old',old),('new',new)]:
    objects=[o for s in frames.values() for o in s['supplemental_instances']]
    profile[name]={'frames':len(frames),'boxes':len(objects),'uncertain':sum(o['uncertain'] for o in objects)}
print(json.dumps(profile,indent=2))

{
  "old": {
    "frames": 33,
    "boxes": 106,
    "uncertain": 13
  },
  "new": {
    "frames": 33,
    "boxes": 96,
    "uncertain": 0
  }
}


## Results

### 3. Reconcile semantic changes without treating reordering as edits

In [3]:
removed=[]; confirmed=[]; geometry_edits=[]; tag_changes=[]; additions=[]
for sid in sorted(old):
    before=old[sid]; after=new[sid]
    if (before['cvat_result'],before['complete_for_size_rule'])!=(after['cvat_result'],after['complete_for_size_rule']):
        tag_changes.append(sid)
    old_objects=before['supplemental_instances']; new_objects=after['supplemental_instances']
    same_ids={o['instance_id'] for o in old_objects}&{o['instance_id'] for o in new_objects}
    unmatched_old=[o for o in old_objects if o['instance_id'] not in same_ids]
    unmatched_new=[o for o in new_objects if o['instance_id'] not in same_ids]
    for a in list(unmatched_old):
        matches=[b for b in unmatched_new if a['bbox_xyxy']==b['bbox_xyxy']]
        if len(matches)==1:
            b=matches[0];confirmed.append({'sample_id':sid,'before':a,'after':b});unmatched_old.remove(a);unmatched_new.remove(b)
    for a in list(unmatched_old):
        matches=[b for b in unmatched_new if sum(x==y for x,y in zip(a['bbox_xyxy'],b['bbox_xyxy']))==3 and a['uncertain']==b['uncertain'] and a['truncated']==b['truncated']]
        if len(matches)==1:
            b=matches[0];geometry_edits.append({'sample_id':sid,'before':a,'after':b});unmatched_old.remove(a);unmatched_new.remove(b)
    removed.extend({'sample_id':sid,**o,'removal_reason':'confirmed non-crater by direct human reply; see review_context.json'} for o in unmatched_old)
    additions.extend({'sample_id':sid,**o} for o in unmatched_new)
assert len(removed)==10 and len(confirmed)==3 and len(geometry_edits)==1 and not additions and not tag_changes
assert all(o['uncertain'] for o in removed)
assert all(o['before']['uncertain'] and not o['after']['uncertain'] for o in confirmed)
print(json.dumps({'removed_by_frame':dict(Counter(o['sample_id'] for o in removed)),'attribute_confirmations':[o['sample_id'] for o in confirmed],'bbox_edit_frames':[o['sample_id'] for o in geometry_edits],'image_review_tags_changed':tag_changes,'unexpected_new_objects':len(additions)},indent=2))
print(json.dumps([{'sample_id':o['sample_id'],'bbox_xyxy':o['bbox_xyxy'],'old_truncated':o['truncated'],'reason':o['removal_reason']} for o in removed],indent=2))

{
  "removed_by_frame": {
    "moon_seg_002": 1,
    "moon_seg_004": 5,
    "moon_seg_026": 1,
    "moon_seg_027": 1,
    "moon_seg_028": 1,
    "moon_seg_030": 1
  },
  "attribute_confirmations": [
    "moon_seg_005",
    "moon_seg_016",
    "moon_seg_030"
  ],
  "bbox_edit_frames": [
    "moon_seg_028"
  ],
  "image_review_tags_changed": [],
  "unexpected_new_objects": 0
}
[
  {
    "sample_id": "moon_seg_002",
    "bbox_xyxy": [
      166.7,
      30.65,
      189.38,
      50.49
    ],
    "old_truncated": false,
    "reason": "confirmed non-crater by direct human reply; see review_context.json"
  },
  {
    "sample_id": "moon_seg_004",
    "bbox_xyxy": [
      215.7,
      222.43,
      256.0,
      252.62
    ],
    "old_truncated": false,
    "reason": "confirmed non-crater by direct human reply; see review_context.json"
  },
  {
    "sample_id": "moon_seg_004",
    "bbox_xyxy": [
      166.46,
      80.0,
      196.35,
      101.81
    ],
    "old_truncated": false,
    "reason

### 4. Verify v3/v4 and explicit human decisions

In [4]:
context=json.loads((WORKSPACE/'data_orbital/crater_review_context_v2.json').read_text(encoding='utf-8'))
assert context['corrected_export']['deletion_reason']=='confirmed_non_craters_by_human_annotator'
assert context['deletion_confirmation']['statement']=='����������, ��� ��� �� �������'
assert len(context['visibility_confirmations'])==3
assert not context['whole_dataset_ground_truth_accepted']
OLD_BUILD=WORKSPACE/'data_orbital/crater_annotations_v3'
BUILD=WORKSPACE/'data_orbital/crater_annotations_v4'
for folder in (OLD_BUILD,BUILD):
    verification=verify_crater_build(WORKSPACE,folder)
    assert verification['integrity_verified'] and verification['reference_ready_questions']==0
    print(json.dumps(verification,indent=2))
metadata=json.loads((BUILD/'metadata.json').read_text(encoding='utf-8'))
questions=read_jsonl(BUILD/'questions.jsonl')
unknown=[q for q in questions if q['reference_status']=='UNKNOWN_REFERENCE']
assert len(questions)==400 and metadata['invariant_candidate_answers']==397 and len(unknown)==3
assert {q['sample_id'] for q in unknown}=={'mars_seg_018'}
assert {q['task_id'] for q in unknown}=={'crater_center_v1','crater_largest_location_v1','crater_half_count_v1'}
assert file_hash(OLD_BUILD/'candidate_requests.jsonl')==file_hash(BUILD/'candidate_requests.jsonl')
old_questions={q['request_id']:q for q in read_jsonl(OLD_BUILD/'questions.jsonl')}
assert all(q['candidate_answer']==old_questions[q['request_id']]['candidate_answer'] for q in questions if old_questions[q['request_id']]['candidate_answer'] is not None)
assert metadata['instances_with_geometry']==3165 and metadata['supplemental_instances']==248
print(json.dumps({'candidate_answers':397,'unknown':3,'accepted_targets':0,'unchanged_model_inputs':True,'old_known_answers_preserved':True},indent=2))

{
  "schema": "crater-annotations-v1",
  "integrity_verified": true,
  "bundle_sha256": "7f3a964b39ea02d3b7053c1dad0071d61cfa275578c50592e7d562a675294a1d",
  "images": 100,
  "candidate_questions": 400,
  "reference_ready_questions": 0,
  "ready_for_scored_inference": false,
  "scientific_result": false
}
{
  "schema": "crater-annotations-v1",
  "integrity_verified": true,
  "bundle_sha256": "2c81ac47fcba3ba1d8867f8f2e1c6fb9d153a220b71c4aa9fa98099dfe44d8a3",
  "images": 100,
  "candidate_questions": 400,
  "reference_ready_questions": 0,
  "ready_for_scored_inference": false,
  "scientific_result": false
}
{
  "candidate_answers": 397,
  "unknown": 3,
  "accepted_targets": 0,
  "unchanged_model_inputs": true,
  "old_known_answers_preserved": true
}


### 5. Verify the retained native band and reference separation

In [5]:
artifacts=json.loads((WORKSPACE/'data_orbital/source_artifacts_v1.json').read_text(encoding='utf-8'))
assert artifacts['parent_bundle_sha256']==metadata['bundle_sha256']
artifact=next(a for a in artifacts['items'] if a['sample_id']=='mars_seg_018')
assert file_hash(WORKSPACE/artifact['image'])==artifact['image_sha256']
sample=next(s for s in read_jsonl(BUILD/'samples.jsonl') if s['sample_id']=='mars_seg_018')
assert sample['source_split']==artifact['source_split']=='val'
assert sample['image_sha256']==artifact['image_sha256'] and sample['mask_sha256']==artifact['mask_sha256']
assert artifact['benchmark_generated_corruption'] is False
assert artifact['roi_is_full_band_extent'] is False and artifact['band_extent_annotation'] is None
assert set(artifact['unknown_reference_tasks'])=={q['task_id'] for q in unknown}
q1=next(q for q in questions if q['sample_id']=='mars_seg_018' and q['task_id']=='crater_presence_v1')
assert q1['candidate_answer']=='YES' and artifact['admitted_scoring_targets']==0
print(json.dumps({'sample':artifact['sample_id'],'original_image_hash_verified':True,'split':'val','role':artifact['experimental_role'],'generated_E2':False,'unknown_questions_preserved':len(unknown)},indent=2))

{
  "sample": "mars_seg_018",
  "original_image_hash_verified": true,
  "split": "val",
  "role": "additional_native_artifact_exploratory_case",
  "generated_E2": false,
  "unknown_questions_preserved": 3
}


## Takeaways

The corrected archive passes native import checks; all 33 Moon val images remain. Its ten deletions are explicitly human-reviewed non-craters, not silent removal of ambiguity. Three limited rim confirmations are hash-bound, and all 365 formerly known candidate answers remain unchanged. Current v4 has 397 candidate answers and 3 unknowns; no scoring targets were auto-accepted. Keep the gray band as an exploratory native artifact with original pixels, source-val split and unknown Q2�Q4 references preserved. The artifact record is not a mask of its complete extent, a synthetic E2 variant, an independent extra image, or evidence of a causal paired robustness effect.

Validation: nbformat checked and ordinary Python cells executed sequentially with captured outputs, not via a Jupyter kernel. Optional additional kernel execution in an environment with nbclient/ipykernel/nbconvert installed: `python -m jupyter nbconvert --execute --to notebook --inplace notebooks/moon_val_revision_v2_audit.ipynb`. No Jupyter packages were installed.